# 2. MCP tools, resources, discovery, and schema boundaries

In [ ]:
print("Embedded dataset and deterministic offline lesson are ready.")

EDUCATIONAL — SELF-CONTAINED

MCP separates an agent from an integration's implementation. We use the real MCP SDK `Tool` and `Resource` schema objects, plus a tiny in-process client that models discovery and calls without a server subprocess. The boundary accepts JSON-shaped arguments and returns JSON-shaped results.

In [ ]:
from mcp.types import Resource, Tool


tools = [
    Tool(
        name="search_recalls",
        description="Find recalls by recall number",
        inputSchema={"type": "object", "properties": {"recall_number": {"type": "string"}}, "required": ["recall_number"]},
    )
]
resources = [
    Resource(uri="recall://policy/traceability", name="traceability policy", mimeType="text/plain")
]

class InProcessMCPClient:
    def list_tools(self) -> list[Tool]:
        return tools

    def list_resources(self) -> list[Resource]:
        return resources

    def call_tool(self, name: str, arguments: dict) -> dict:
        if name != "search_recalls":
            raise KeyError(name)
        schema = tools[0].input_schema
        required = schema["required"]
        if any(key not in arguments for key in required):
            raise ValueError("schema boundary: recall_number is required")
        return {"recall_number": arguments["recall_number"], "hazard": "Salmonella", "origin": "PUBLIC_SNAPSHOT"}

    def read_resource(self, uri: str) -> str:
        if uri != str(resources[0].uri):
            raise KeyError(uri)
        return "Keep source citations with every observation."

client = InProcessMCPClient()
print("discovered tools ->", [(tool.name, tool.input_schema) for tool in client.list_tools()])
print("discovered resources ->", [(str(resource.uri), resource.name) for resource in client.list_resources()])
recall = client.call_tool("search_recalls", {"recall_number": "H-1230-2026"})
print("tool result ->", recall)
print("resource result ->", client.read_resource("recall://policy/traceability"))
assert recall["hazard"] == "Salmonella"
try:
    client.call_tool("search_recalls", {})
except ValueError as error:
    print("rejected invalid arguments ->", error)
else:
    raise AssertionError("invalid arguments crossed the schema boundary")
print("ASSERTION PASSED: discovery, resource reading, and schema validation are visible")


In [ ]:
print("EDUCATIONAL — SELF-CONTAINED")
print("A tool is callable capability; a resource is readable context; schemas are the boundary.")
assert client.list_tools()[0].name == "search_recalls"
assert str(client.list_resources()[0].uri).startswith("recall://")
print("ASSERTION PASSED: EDUCATIONAL — SELF-CONTAINED")